In [1]:
import json
import os
import csv

In [2]:
from kafka import KafkaProducer

In [3]:
producer = KafkaProducer(
    bootstrap_servers=['kafka:9092'],
    api_version=(3, 7, 0),
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

In [4]:
KAFKA_TOPIC = 'raw_data'
KAFKA_SERVER = 'kafka:9092'
DATA_DIR = '/home/jovyan/data'

In [5]:
def cast_row(row):
    int_fields = ['id', 'customer_age', 'product_quantity', 'sale_customer_id',
                  'sale_seller_id', 'sale_product_id', 'sale_quantity', 'product_reviews']
    float_fields = ['product_price', 'sale_total_price', 'product_weight', 'product_rating']

    for k, v in row.items():
        if v == '' or v is None:
            row[k] = None
        elif k in int_fields:
            row[k] = int(v)
        elif k in float_fields:
            row[k] = float(v)
    return row

In [6]:
def send_data():
    files = ['MOCK_DATA.csv'] + [f'MOCK_DATA ({i}).csv' for i in range(1, 10)]
    for file_name in files:
        path = os.path.join(DATA_DIR, file_name)
        if not os.path.exists(path): continue

        with open(path, mode='r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                producer.send(KAFKA_TOPIC, value=cast_row(row))
        print(f"Sent: {file_name}")
    producer.flush()

In [7]:
send_data()

Sent: MOCK_DATA.csv
Sent: MOCK_DATA (1).csv
Sent: MOCK_DATA (2).csv
Sent: MOCK_DATA (3).csv
Sent: MOCK_DATA (4).csv
Sent: MOCK_DATA (5).csv
Sent: MOCK_DATA (6).csv
Sent: MOCK_DATA (7).csv
Sent: MOCK_DATA (8).csv
Sent: MOCK_DATA (9).csv
